In [ ]:
import copy

import numpy as np
from torch.linalg import slogdet

from tqdm import trange
import matplotlib.pyplot as plt

import adaptive_latents
from adaptive_latents import StreamingKalmanFilter, Bubblewrap, ArrayWithTime, Pipeline, proSVD, Tee, VJF, CenteringEstimator
from adaptive_latents.regressions import BaseKNearestNeighborRegressor
from adaptive_latents.stim_regressor import StimRegressor
from tqdm.autonotebook import tqdm

rng = np.random.default_rng(0)


In [ ]:
d = adaptive_latents.datasets.Odoherty21Dataset()

In [ ]:
def high_d_S(low_d_point, high_d_stim):
    return high_d_stim

def S(low_d_point, high_d_stim, pro):
    return pro.transform(high_d_S(low_d_point, high_d_stim)[None,:])


In [ ]:
def do_experiment(input_arrays, decay_rate = .9, stim_scale=1, rng=None, s_inputs_to_evaluate_on=None):
    if rng is None:
        rng = np.random.default_rng(0)
    centerer = CenteringEstimator()

    pro = proSVD(k=10)

    sr = StimRegressor(
        autoreg=StreamingKalmanFilter(),
        stim_reg=BaseKNearestNeighborRegressor(k=1, maxlen=1000),
        attempt_correction=True
    )

    stims = []
    predictions = []
    latents = []
    dt_X = []
    s_hat_evals = []
    s_eval = []

    s_inputs_evaluated_on = []

    to_add = np.zeros(input_arrays[0].shape[1])

    for input_array in input_arrays:
        pbar = tqdm(total=round(input_array.t[-1],2))
        for data in Pipeline().streaming_run_on(input_array):

            latent_location = pro.transform(centerer.transform(data))

            stim = np.zeros(data.shape[1])
            if rng.random() < .05 and pro.is_initialized:
                stim = pro.Q[:,0] * stim_scale
                to_add += high_d_S(latent_location, stim)

            if s_inputs_to_evaluate_on == 'record' and np.any(stim):
                s_inputs_evaluated_on.append(copy.deepcopy((latent_location, stim, pro)))
                s_eval.append(S(latent_location, stim, pro))

            data = data + to_add
            to_add = to_add * decay_rate

            data = centerer.step(data)
            data = pro.step(data)
            latents.append(data)

            qX = ArrayWithTime([[1]], data.t)
            sr.step(np.array([[stim]]), stream='stim')
            prediction = sr.step(qX, stream='dt_X')
            sr.step(data, stream='X')



            stims.append(ArrayWithTime(stim, data.t))
            predictions.append(prediction)
            dt_X.append(qX)

            if not isinstance(s_inputs_to_evaluate_on, str) and s_inputs_to_evaluate_on is not None and np.any(stim):
                point_evals = []
                for latent_location, stim, _ in s_inputs_to_evaluate_on:
                    stim_reg_input = np.hstack([latent_location.flatten(), stim.flatten()])
                    diff = sr.stim_reg.predict(stim_reg_input)
                    point_evals.append(diff.flatten())
                s_hat_evals.append(ArrayWithTime(point_evals,data.t))

                if not s_eval:
                    for inputs in s_inputs_to_evaluate_on:
                        s_eval.append(S(*inputs))


            if not sr.autoreg.get_parameter_fitting_state():
                sr.autoreg.set_parameter_fitting_state(True)

            pbar.update(round(data.t,2) - pbar.n)
        sr.autoreg.set_parameter_fitting_state(False)


    stims = ArrayWithTime.from_list(stims)
    predictions = ArrayWithTime.from_list(predictions, drop_early_nans=True, squeeze_type='to_2d')
    latents = ArrayWithTime.from_list(latents, squeeze_type='to_2d')
    s_hat_evals = ArrayWithTime.from_list(s_hat_evals, drop_early_nans=True)
    s_eval = np.squeeze(s_eval)
    return (predictions, latents, stims), (s_hat_evals, s_eval, s_inputs_evaluated_on)



In [ ]:
stim_scale = 20
(record_predictions, record_latents, record_stims), (record_s_hat_evals, record_s_eval, record_s_inputs_evaluated_on) = do_experiment([d.neural_data], stim_scale=stim_scale, s_inputs_to_evaluate_on='record')

In [ ]:
(eval_predictions, eval_latents, eval_stims), (eval_s_hat_evals, eval_s_eval, eval_s_inputs_evaluated_on) = do_experiment([d.neural_data], stim_scale=stim_scale, s_inputs_to_evaluate_on=record_s_inputs_evaluated_on)



In [ ]:

assert np.array_equal(record_predictions, eval_predictions, equal_nan=True)
assert np.array_equal(record_latents, eval_latents, equal_nan=True)
assert np.array_equal(record_stims, eval_stims, equal_nan=True)
assert np.array_equal(record_s_eval, eval_s_eval, equal_nan=True)
assert record_s_hat_evals.size == 0
assert len(eval_s_inputs_evaluated_on) == 0
predictions, latents, stims, s_hat_evals, s_eval = eval_predictions, eval_latents, eval_stims, eval_s_hat_evals, record_s_eval



In [ ]:
print(f"evaluated on {s_hat_evals.shape[0]} timepoints; {s_hat_evals.shape[1]} test points with {s_hat_evals.shape[2]}-d returns")
error = np.linalg.norm(s_hat_evals - s_eval, axis=2).mean(axis=1)
error = ArrayWithTime.from_transformed_data(error, s_hat_evals)
pred_zero_error = np.linalg.norm(s_hat_evals*0 - s_eval, axis=2).mean(axis=1)[0]


In [ ]:
%matplotlib inline
rig, ax = plt.subplots()
ax.scatter(error.t, error, s=2)
ax.axhline(pred_zero_error, linestyle='--', color='k')
ax.set_ylim([0, ax.get_ylim()[1]])

In [ ]:
%matplotlib qt
plt.matshow(stims[stims.any(axis=1)])

In [ ]:
predictions.shape

In [ ]:
%matplotlib qt
fig, ax = plt.subplots()

i = 0
ax.plot(latents.t, latents[:,i])
ax.plot(predictions.t, predictions[:,0,i])
